# Raipur 30-zone forecasting (XGBoost) — Yuva Yodha Challenge 03
Trains **XGBoost** models for **demand** and **solar** forecasters for 15 / 30 / 45 / 60 minutes ahead on the v2 synthetic dataset, benchmarks them against naive baselines on the unseen April test period, and exports forecasts for the MILP / MPC scheduler.

**Run order:** run all cells top to bottom (Runtime → Run all). A CPU runtime is enough; expect roughly 5–15 minutes.

In [ ]:
import os, json, zipfile, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
import xgboost as xgb
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

BASE     = os.environ.get("WORK", "/content")
DATA_DIR = f"{BASE}/raipur_v2"
OUT_DIR  = f"{BASE}/outputs"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(f"{OUT_DIR}/models", exist_ok=True)
SEED = 42

## 1. Load the dataset
Upload `raipur_30zone_v2.zip` when prompted (or put the extracted CSVs in the `DATA_DIR` folder).

In [ ]:
if not os.path.exists(f"{DATA_DIR}/ML_train_features.csv"):
    from google.colab import files
    up = files.upload()                       # choose raipur_30zone_v2.zip
    zipfile.ZipFile(list(up.keys())[0]).extractall(DATA_DIR)

rd = lambda n: pd.read_csv(f"{DATA_DIR}/ML_{n}_features.csv", parse_dates=["Timestamp"])
tr, va, te = rd("train"), rd("validation"), rd("test")
zm = pd.read_csv(f"{DATA_DIR}/zone_master.csv")[["Zone_ID", "Zone_Type"]]
for n, d in (("train", tr), ("validation", va), ("test", te)):
    print(f"{n:11s} {d.Timestamp.min()} -> {d.Timestamp.max()}  rows={len(d):,}")

# --- safety checks: chronological order, no leaky columns ---
assert tr.Timestamp.max() < va.Timestamp.min() and va.Timestamp.max() < te.Timestamp.min(), "splits overlap"
leaky = [c for c in tr.columns if any(k in c.upper() for k in ("BASELINE", "UNSERVED", "SOC", "SCHEDULE", "GRID_AVAIL"))]
assert not leaky, leaky
print("OK: chronological split, no leaky columns")

## 2. Features and targets

In [ ]:
ID       = ["Timestamp", "Zone_ID"]
TARGETS  = [c for c in tr.columns if c.startswith("y_")]
FEATURES = [c for c in tr.columns if c not in ID + TARGETS]
H        = {1: "15 min", 2: "30 min", 3: "45 min", 4: "60 min"}
cap      = lambda d: d.Rooftop_Solar_Capacity_MW + d.Community_Solar_Capacity_MW
print(len(FEATURES), "features:", FEATURES)

## 3. Train
* **Demand**: target is `log(demand(t+h) / demand(t))`. Trees cannot extrapolate levels, and demand trends up from January to April, so predicting the *ratio* is more robust than predicting MW directly.
* **Solar**: target is output as a fraction of the zone's installed PV; forced to 0 when clear-sky irradiance at t+h is zero (night).
* Early stopping on the **validation** set. The test set is not touched until section 4.

In [ ]:
params = dict(n_estimators=3000, learning_rate=0.05, max_depth=7, min_child_weight=50, subsample=0.8,
              colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist", eval_metric="mae",
              early_stopping_rounds=100, random_state=SEED, n_jobs=-1)
def fit(Xtr, ytr, Xva, yva):
    m = xgb.XGBRegressor(**params)
    return m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)

models, best_iter = {}, {}
for h in H:
    # demand
    m = fit(tr[FEATURES], np.log(tr[f"y_demand_h{h}"] / tr.Demand_MW),
            va[FEATURES], np.log(va[f"y_demand_h{h}"] / va.Demand_MW))
    models[f"demand_h{h}"] = m; best_iter[f"demand_h{h}"] = int(m.best_iteration)
    # solar
    m = fit(tr[FEATURES], tr[f"y_solar_h{h}"] / cap(tr), va[FEATURES], va[f"y_solar_h{h}"] / cap(va))
    models[f"solar_h{h}"] = m; best_iter[f"solar_h{h}"] = int(m.best_iteration)
    print(f"h={h} ({H[h]}) trained | best iters  demand={best_iter[f'demand_h{h}']}  solar={best_iter[f'solar_h{h}']}")

## 4. Evaluate on the unseen test period (13–30 April)

In [ ]:
def predict(d, h):
    pd_ = d.Demand_MW * np.exp(models[f"demand_h{h}"].predict(d[FEATURES]))
    ps  = np.clip(models[f"solar_h{h}"].predict(d[FEATURES]), 0, None) * cap(d)
    ps  = np.where(d[f"Clearsky_GHI_h{h}"] > 0, ps, 0.0)
    return pd_.values, ps

mape = lambda y, p: 100 * np.mean(np.abs(y - p) / y)
rows, preds = [], te[ID].copy()
for h in H:
    pdem, psol = predict(te, h)
    preds[f"pred_demand_h{h}"], preds[f"pred_solar_h{h}"] = pdem, psol
    yd, ys = te[f"y_demand_h{h}"], te[f"y_solar_h{h}"]
    day = te[f"Clearsky_GHI_h{h}"] > 50
    csr = te[f"Clearsky_GHI_h{h}"] / te.Clearsky_GHI_Wm2.where(te.Clearsky_GHI_Wm2 > 50)
    smart = (te.Total_Local_Solar_MW * csr).fillna(te.Total_Local_Solar_MW)       # clear-sky-scaled persistence
    nmae = lambda p: 100 * np.mean(np.abs(ys[day] - p[day]) / cap(te)[day])
    rows.append({"horizon": H[h],
        "demand MAPE persistence %": mape(yd, te.Demand_MW), "demand MAPE model %": mape(yd, pdem),
        "solar nMAE persistence %": nmae(te.Total_Local_Solar_MW), "solar nMAE smart-persist %": nmae(smart),
        "solar nMAE model %": nmae(pd.Series(psol, index=te.index))})
res = pd.DataFrame(rows).set_index("horizon").round(2)
print(res.to_string())
print("\nsolar nMAE = mean absolute error as % of installed PV capacity, daylight intervals only")

In [ ]:
# Same thing by zone type (60 min ahead) and for the regional total
t = te.merge(zm, on="Zone_ID").assign(pd_=preds.pred_demand_h4.values)
by = t.groupby("Zone_Type").apply(lambda g: pd.Series({"persistence MAPE %": mape(g.y_demand_h4, g.Demand_MW),
                                                      "model MAPE %": mape(g.y_demand_h4, g.pd_)})).round(2)
print(by.to_string())
reg = t.groupby("Timestamp").agg(y=("y_demand_h4", "sum"), p=("pd_", "sum"), now=("Demand_MW", "sum"))
print(f"\nRegional total, 60 min ahead: persistence {mape(reg.y, reg.now):.2f}%  model {mape(reg.y, reg.p):.2f}%")

In [ ]:
ZONE = "Z21"                                   # change to any Z01..Z30
z = te[te.Zone_ID == ZONE].reset_index(drop=True); pz = preds[preds.Zone_ID == ZONE].reset_index(drop=True)
wk = slice(0, 96 * 7)
fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax[0].plot(z.Timestamp[wk], z.y_demand_h4[wk], label="actual"); ax[0].plot(z.Timestamp[wk], pz.pred_demand_h4[wk], label="forecast (t+60 min)")
ax[0].set_ylabel("Demand (MW)"); ax[0].legend(); ax[0].set_title(f"{ZONE}: demand, first test week")
ax[1].plot(z.Timestamp[wk], z.y_solar_h4[wk], label="actual"); ax[1].plot(z.Timestamp[wk], pz.pred_solar_h4[wk], label="forecast (t+60 min)")
ax[1].set_ylabel("Solar (MW)"); ax[1].legend(); plt.tight_layout(); plt.show()

imp = pd.Series(models["demand_h4"].get_booster().get_score(importance_type="gain")).reindex(FEATURES).fillna(0).sort_values().tail(12)
imp.plot.barh(figsize=(7, 4), title="Demand model (t+60): top features by gain"); plt.tight_layout(); plt.show()

## 5. Save models and export forecasts for the optimizer

In [ ]:
for k, m in models.items(): joblib.dump(m, f"{OUT_DIR}/models/{k}.joblib")
json.dump({"features": FEATURES, "best_iteration": best_iter, "demand_target": "log(y/Demand_MW)",
           "solar_target": "y/(rooftop+community capacity)"}, open(f"{OUT_DIR}/models/meta.json", "w"), indent=2)

# Forecast file: what the MILP/MPC layer consumes (one row per zone per 15-min step; each row forecasts t+15 ... t+60)
fc = preds.merge(te[ID + ["Demand_MW", "Total_Local_Solar_MW"]], on=ID)
fc.to_csv(f"{OUT_DIR}/forecasts_test_xgb.csv", index=False, float_format="%.4f")
res.to_csv(f"{OUT_DIR}/test_metrics_xgb.csv")
print(fc.shape, "->", f"{OUT_DIR}/forecasts_test_xgb.csv")

try:
    import shutil; from google.colab import files
    shutil.make_archive(f"{BASE}/raipur_forecasting_xgb_outputs", "zip", OUT_DIR)
    files.download(f"{BASE}/raipur_forecasting_xgb_outputs.zip")
except ImportError:
    pass

## 6. (Optional) Refit on train + validation for final use
Only do this **after** you have reported the test numbers above. It reuses the best iteration counts, so no further tuning is involved.

In [ ]:
full = pd.concat([tr, va])
final = {}
for h in H:
    for kind in ("demand", "solar"):
        p = {**params, "n_estimators": best_iter[f"{kind}_h{h}"] + 1, "early_stopping_rounds": None}
        y = np.log(full[f"y_demand_h{h}"] / full.Demand_MW) if kind == "demand" else full[f"y_solar_h{h}"] / cap(full)
        final[f"{kind}_h{h}"] = xgb.XGBRegressor(**p).fit(full[FEATURES], y)
        joblib.dump(final[f"{kind}_h{h}"], f"{OUT_DIR}/models/final_{kind}_h{h}.joblib")
print("saved final models")

## 7. (Optional) Head-to-head with LightGBM, 60 minutes ahead
Trains the two 60-minute LightGBM models with the same early-stopping rule so you can quote a fair comparison.

In [ ]:
import lightgbm as lgb
def fit_lgb(Xtr, ytr, Xva, yva):
    m = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.04, num_leaves=63, min_child_samples=50, subsample=0.8,
                          subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbose=-1)
    return m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="l1", callbacks=[lgb.early_stopping(100, verbose=False)])
md_ = fit_lgb(tr[FEATURES], np.log(tr.y_demand_h4 / tr.Demand_MW), va[FEATURES], np.log(va.y_demand_h4 / va.Demand_MW))
ms_ = fit_lgb(tr[FEATURES], tr.y_solar_h4 / cap(tr), va[FEATURES], va.y_solar_h4 / cap(va))
ld = te.Demand_MW * np.exp(md_.predict(te[FEATURES]))
ls = np.where(te.Clearsky_GHI_h4 > 0, np.clip(ms_.predict(te[FEATURES]), 0, None) * cap(te), 0.0)
day = te.Clearsky_GHI_h4 > 50
nm = lambda p: 100 * np.mean(np.abs(te.y_solar_h4[day] - np.asarray(p)[day.values]) / cap(te)[day])
print(pd.DataFrame({"XGBoost": [mape(te.y_demand_h4, preds.pred_demand_h4), nm(preds.pred_solar_h4)],
                    "LightGBM": [mape(te.y_demand_h4, ld), nm(ls)]},
                   index=["demand MAPE % (60 min)", "solar nMAE % (60 min)"]).round(2).to_string())

## What you can and cannot claim
* This is a **synthetic** dataset: zone types, weather, faults and feeder limits are assumptions. Say *"validated on a synthetic 30-zone dataset calibrated to the Raipur regional load; the pipeline accepts real feeder telemetry and weather"*.
* In production, `Temperature_C`, `Cloud_Cover_Pct`, `GHI_Wm2` would come from a **weather forecast**, not observations at time t. Retrain with forecast inputs once you have them.
* Next step: feed `forecasts_test_xgb.csv` into the MILP/MPC scheduler and measure outage-hours and unserved energy against the `BASELINE_*` columns in the `raw_*.csv` files.